### ROUGH DRAFT NOTEBOOK FOR FORECASTING MODEL

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt


pd.set_option('display.max_columns', None)

In [2]:
path = Path('projections')
projects = {}

for file in sorted(path.glob('*.csv'), key=lambda p: p.name):
    current_df = pd.read_csv(file)
    file_name = file.stem
    projects[file_name] = current_df

In [ ]:
# Some formatting/ cleaning up stuff
for name, df in projects.items():
    df_cleaned = df.iloc[:, 1:].copy()

    if len(df_cleaned) > 0:
        df_cleaned.iat[-1, 0] = "Total"

        start_col_idx = 6
        if df_cleaned.shape[1] > start_col_idx:
            last_row = df_cleaned.iloc[[-1], start_col_idx:]
            
            # Clean string formatting, remove commas, handle parentheses, and cast to float
            cleaned_last_row = (
                last_row.astype("string")
                .apply(lambda s: s.str.split(r"\r\n|\n|\r", regex=True).str[0])
                .apply(lambda s: s.str.replace(r',', '', regex=True))
                .apply(lambda s: s.str.replace(r'^\((.*)\)$', r'-\1', regex=True))
                .astype(float)
            )
            
            # Assign it back
            df_cleaned.iloc[-1, start_col_idx:] = cleaned_last_row.iloc[0]

    df_cleaned = df_cleaned.reset_index(drop=True)
    projects[name] = df_cleaned

TypeError: Invalid value '39741341.68' for dtype 'str'. Value should be a string or missing value, got 'float' instead.

In [ ]:
timeline_dates = pd.date_range(start='2011-01-01', end='2027-06-01', freq='MS')
master_timeline = timeline_dates.strftime('%b %Y').tolist()

for name, df in projects.items():
    static_columns = list(df.columns[:7])
    all_desired_columns = static_columns + master_timeline
    df_unified = df.reindex(columns=all_desired_columns, fill_value=0)
    projects[name] = df_unified

In [ ]:
# Drop all categories and leave only totals
for name, df in projects.items():
    total_only = df[df.iloc[:, 0].astype(str).eq("Total")].copy()
    total_only = total_only.drop(columns=["Line Item", "Description"])
    projects[name] = total_only.reset_index(drop=True)

In [ ]:
# Add project code as first column
for name, df in projects.items():
    if df.empty:
        continue

    if "Project Code" in df.columns:
        df = df.drop(columns=["Project Code"])
    df.insert(0, "Project Code", name)

    projects[name] = df

In [ ]:
# Squish all projects into a single dataframe
non_empty_projects = [df for df in projects.values() if not df.empty]
all_projects_df = pd.concat(non_empty_projects, ignore_index=True)

In [ ]:
import os

# Remove "Actuals" columns (I don't think we need these?)
cols_to_drop = ["Actuals To Date", "Actuals + Projections"]
all_projects_df = all_projects_df.drop(columns=cols_to_drop, errors="ignore")

In [ ]:
# Convert monthly spend to cumulative sum
timeline_cols = all_projects_df.columns[5:]

all_projects_df[timeline_cols] = (
    all_projects_df[timeline_cols]
    .astype(str)
    .replace({',': '', r'\(': '-', r'\)': ''}, regex=True)
    .astype(float)
)

all_projects_df[timeline_cols] = all_projects_df[timeline_cols].cumsum(axis=1)
all_projects_df.head(20)

In [ ]:
# Remove project 5149 (no spending?)
all_projects_df = all_projects_df[all_projects_df["Project Code"].astype(str) != "5149"].reset_index(drop=True)

In [ ]:
timeline_cols = all_projects_df.columns[5:]

# Plot first 3 S_curves using cumulative sums
for index, row in all_projects_df.head(3).iterrows():
    project_code = row['Project Code']
    cumulative_values = row[timeline_cols]
    plt.figure(figsize=(10, 4))
    plt.plot(timeline_cols, cumulative_values, linewidth=2, color='#1f77b4')
    plt.title(f"Cumulative Expenditure S-Curve: Project {project_code}", fontsize=14)
    plt.xlabel("Timeline", fontsize=12)
    plt.ylabel("Cumulative Cost ($)", fontsize=12)
    plt.xticks(timeline_cols[::12], rotation=45) 
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
########################################################
"""

Deconstructs timelines into parameters "Start_Date", "Stop_Date", "S_Curve_L" (Max Value),
"S_Curve_k" (Growth Rate), and "S_Curve_t0" (Midpoint), then reconstructs S-curves using 
these parameters (instead of counting cumulative sums)

This version is intended as a proof-of-concept, to show how the parameterization/reconstruction
works. Since in the actual model we're not predicting things like when the project starts, we're
gonna use the version in the next cell

"""
########################################################

all_projects_df_1 = all_projects_df.copy()

# Define our S-curve mathematical model
def logistic_curve(t, L, k, t0):
    return L / (1 + np.exp(-k * (t - t0)))

# Prepare our X-axis as numeric time steps
timeline_cols = all_projects_df_1.columns[5:]
x_data = np.arange(len(timeline_cols))

# Create empty lists to store all our parameters
L_params = []
k_params = []
t0_params = []
start_dates = []
stop_dates = []

# Loop through the projects
for index, row in all_projects_df_1.iterrows():
    y_data = row[timeline_cols].values.astype(float)
    start_idx = np.argmax(y_data > 0)
    stop_idx = np.argmax(y_data == y_data[-1])
    start_dates.append(timeline_cols[start_idx])
    stop_dates.append(timeline_cols[stop_idx])

    # Fit the Logistic Curve (same as before)
    initial_guess = [y_data[-1], 0.1, len(x_data)/2]
    popt, pcov = curve_fit(logistic_curve, x_data, y_data, p0=initial_guess, maxfev=1000000)
        
    L_params.append(popt[0])
    k_params.append(popt[1])
    t0_params.append(popt[2])

# Save all these features as new columns in our dataframe
all_projects_df_1['S_Curve_L'] = L_params
all_projects_df_1['S_Curve_k'] = k_params
all_projects_df_1['S_Curve_t0'] = t0_params
all_projects_df_1['Start_Date'] = start_dates
all_projects_df_1['Stop_Date'] = stop_dates

# Check results in table
display(all_projects_df_1[['Project Code', 'Start_Date', 'Stop_Date', 'S_Curve_L', 'S_Curve_k', 'S_Curve_t0']].head(60))

# Set up our timeline and X-axis indices
timeline_dates = pd.date_range(start='2011-01-01', end='2027-06-01', freq='MS')
timeline_cols = timeline_dates.strftime('%b %Y').tolist()
x_data = np.arange(len(timeline_cols))

# Loop through the first 3 projects
for index, row in all_projects_df_1.head(60).iterrows():
    L = row['S_Curve_L']
    k = row['S_Curve_k']
    t0 = row['S_Curve_t0']
    start = row['Start_Date']
    stop = row['Stop_Date']
        
    # Generate curve using parameters
    predicted_y = logistic_curve(x_data, L, k, t0)
    plt.figure(figsize=(10, 4))
    plt.plot(timeline_cols, predicted_y, linewidth=3, color='#ff7f0e', label="Parameterized S-Curve")
    
    # Formatting
    if pd.notna(start) and pd.notna(stop):
        start_idx = timeline_cols.index(start)
        stop_idx = timeline_cols.index(stop)
        plt.axvline(x=start_idx, color='green', linestyle='--', alpha=0.7, label=f"Start: {start}")
        plt.axvline(x=stop_idx, color='red', linestyle='--', alpha=0.7, label=f"Stop: {stop}")

    plt.title(f"Reconstructed S-Curve from Parameters: Project {row['Project Code']}", fontsize=14)
    plt.xlabel("Timeline", fontsize=12)
    plt.ylabel("Cumulative Cost ($)", fontsize=12)
    plt.xticks(timeline_cols[::12], rotation=45) 
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
########################################################
"""

Same thing as above but uses a duration variable instead to express length of the project. All 
projects "start" in the same arbitrary place. This is what the model will actually use

"""
########################################################

all_projects_df_2 = all_projects_df.copy()

# Prepare timeline
timeline_dates = pd.date_range(start='2011-01-01', end='2027-06-01', freq='MS')
timeline_cols = timeline_dates.strftime('%b %Y').tolist()

L_params = []
k_params = []
t0_params = []
durations = []

# Loop through the projects
for index, row in all_projects_df_2.iterrows():
    y_data_full = row[timeline_cols].values.astype(float)
    start_idx = np.argmax(y_data_full > 0)
    stop_idx = np.argmax(y_data_full == y_data_full[-1])
        
    # Get duration
    num_months = stop_idx - start_idx + 1
    durations.append(num_months)
        
    # Filter for projects less than 3 months long (if they exist)
    if num_months < 3:
        L_params.append(np.nan)
        k_params.append(np.nan)
        t0_params.append(np.nan)
        continue
            
    y_active = y_data_full[start_idx : stop_idx + 1]
    x_active = np.arange(num_months)
        
    # Fit curves
    initial_guess = [y_active[-1], 0.1, num_months / 2]
    popt, pcov = curve_fit(logistic_curve, x_active, y_active, p0=initial_guess, maxfev=1000000)
    
    L_params.append(popt[0])
    k_params.append(popt[1])
    t0_params.append(popt[2])

# Save all these features as new columns in our dataframe
all_projects_df_2['Duration_Months'] = durations
all_projects_df_2['S_Curve_L'] = L_params
all_projects_df_2['S_Curve_k'] = k_params
all_projects_df_2['S_Curve_t0'] = t0_params

# Check results in table
display(all_projects_df_2[['Project Code', 'Duration_Months', 'S_Curve_L', 'S_Curve_k', 'S_Curve_t0']].head(60))

# Loop through and plot frist 3 projects
for index, row in all_projects_df_2.head(60).iterrows():
    L = row['S_Curve_L']
    k = row['S_Curve_k']
    t0 = row['S_Curve_t0']
    duration = row['Duration_Months']
    
    # Generate X axis
    x_data = np.arange(int(duration))
    predicted_y = logistic_curve(x_data, L, k, t0)
    
    # Plot the curve
    plt.figure(figsize=(9, 4))
    plt.plot(x_data, predicted_y, linewidth=3, color='#2ca02c', label="Parameterized S-Curve")
    plt.title(f"Standardized S-Curve: Project {row['Project Code']}", fontsize=14)
    plt.xlabel("Months from Project Start", fontsize=12) 
    plt.ylabel("Cumulative Cost ($)", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
timeline_dates = pd.date_range(start='2011-01-01', end='2027-06-01', freq='MS')
timeline_cols = timeline_dates.strftime('%b %Y').tolist()

# Replace timeline monthly columns with parameters
all_projects_df = all_projects_df_2.drop(columns=timeline_cols, errors='ignore')

In [ ]:
# Remove projects 12, 29 and 42 from dataframe (no data - only ~1-2mo in length)
all_projects_df = all_projects_df[all_projects_df["Project Code"].astype(str) != "W5678"].reset_index(drop=True)
all_projects_df = all_projects_df[all_projects_df["Project Code"].astype(str) != "5570"].reset_index(drop=True)
all_projects_df = all_projects_df[all_projects_df["Project Code"].astype(str) != "5454"].reset_index(drop=True)

# Convert 5 features to float
all_projects_df[['Gross Sq Footage', 'Projected Budget', 'Projected Commitments', 'Estimate at Completion']] = all_projects_df[['Gross Sq Footage', 'Projected Budget', 'Projected Commitments', 'Estimate at Completion']].replace({'[\$,]': ''}, regex=True).astype(float)
all_projects_df.head(60)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV

In [ ]:
########################################################
"""

RANDOM FORESTS FOR EACH: Duration, S_Curve_L (Max Value), S_Curve_k (Growth Rate), S_Curve_t0
INTENTION AT THIS STAGE: Compare results and observe potential bottlenecks for model performance

"""
########################################################

In [ ]:
### Duration: ###
print("DURATION IN MONTHS")
print("")

# Define X, Y and train_test_split
X1 = all_projects_df[['Gross Sq Footage', 'Projected Budget', 'Projected Commitments', 'Estimate at Completion']]
y1 = all_projects_df['Duration_Months']
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=99)

# Display ACTUAL labels
print("Actual values for y_test")
print(y1_test.head(20))

# Train and tune Hyperparameters
rf_single1 = RandomForestRegressor(random_state=99)

param_grid1 = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 1.0]
}

grid_search1 = GridSearchCV(estimator=rf_single1, param_grid=param_grid1, 
                           cv=5, n_jobs=-1, scoring='neg_root_mean_squared_error')

print("Starting hyperparameter tuning...")
grid_search1.fit(X1_train, y1_train)

best_rf_model1 = grid_search1.best_estimator_

print("\n--- Tuning Complete ---")
print(f"Best Hyperparameters: {grid_search1.best_params_}")
print("")
best_rf_model1.fit(X1_train, y1_train)

# Predict and Evaluate
predictions1 = best_rf_model1.predict(X1_test)
print("Predicted values for y_test")
print(predictions1)

# Calculate the error
mse1 = mean_squared_error(y1_test, predictions1)
rmse1 = np.sqrt(mse1)
print("")
print(f"RMSE for Duration_Months: {rmse1:.4f}")
print("-------------------------------------")

In [ ]:
### S_Curve_L: ###
print("MAX VALUE (S_Curve_L)")
print("")

# Define X, Y and train_test_split
X2 = all_projects_df[['Gross Sq Footage', 'Projected Budget', 'Projected Commitments', 'Estimate at Completion']]
y2 = all_projects_df['S_Curve_L']
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=99)

# Display ACTUAL labels
print("Actual values for y_test")
print(y2_test.head(20))

# Train and tune Hyperparameters
rf_single2 = RandomForestRegressor(random_state=99)

param_grid2 = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 1.0]
}

grid_search2 = GridSearchCV(estimator=rf_single2, param_grid=param_grid2, 
                           cv=5, n_jobs=-1, scoring='neg_root_mean_squared_error')

print("Starting hyperparameter tuning...")
grid_search2.fit(X2_train, y2_train)

best_rf_model2 = grid_search2.best_estimator_

print("\n--- Tuning Complete ---")
print(f"Best Hyperparameters: {grid_search2.best_params_}")
print("")
best_rf_model2.fit(X2_train, y2_train)

# Predict and Evaluate
predictions2 = best_rf_model2.predict(X2_test)
print("Predicted values for y_test")
print(predictions2)

# Calculate the error
mse2 = mean_squared_error(y2_test, predictions2)
rmse2 = np.sqrt(mse2)
print("")
print(f"RMSE for Max Value: {rmse2:.4f}")
print("-------------------------------------")

In [ ]:
### S_Curve_k: ###
print("Growth Rate (S_Curve_k)")
print("")

# Define X, Y and train_test_split
X3 = all_projects_df[['Gross Sq Footage', 'Projected Budget', 'Projected Commitments', 'Estimate at Completion']]
y3 = all_projects_df['S_Curve_k']
X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.2, random_state=99)

# Display ACTUAL labels
print("Actual values for y_test")
print(y3_test.head(20))

# Train and tune Hyperparameters
rf_single3 = RandomForestRegressor(random_state=99)

param_grid3 = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 1.0]
}

grid_search3 = GridSearchCV(estimator=rf_single3, param_grid=param_grid3, 
                           cv=5, n_jobs=-1, scoring='neg_root_mean_squared_error')

print("Starting hyperparameter tuning...")
grid_search3.fit(X3_train, y3_train)

best_rf_model3 = grid_search3.best_estimator_

print("\n--- Tuning Complete ---")
print(f"Best Hyperparameters: {grid_search3.best_params_}")
print("")
best_rf_model3.fit(X3_train, y3_train)

# Predict and Evaluate
predictions3 = best_rf_model3.predict(X3_test)
print("Predicted values for y_test")
print(predictions3)

# Calculate the error
mse3 = mean_squared_error(y3_test, predictions3)
rmse3 = np.sqrt(mse3)
print("")
print(f"RMSE for Growth Rate: {rmse3:.4f}")
print("-------------------------------------")

In [ ]:
### S_Curve_t0: ###
print("Curve Midpoint (S_Curve_t0)")
print("")

# Define X, Y and train_test_split
X4 = all_projects_df[['Gross Sq Footage', 'Projected Budget', 'Projected Commitments', 'Estimate at Completion']]
y4 = all_projects_df['S_Curve_t0']
X4_train, X4_test, y4_train, y4_test = train_test_split(X4, y4, test_size=0.2, random_state=99)

# Display ACTUAL labels
print("Actual values for y_test")
print(y4_test.head(20))

# Train and tune Hyperparameters
rf_single4 = RandomForestRegressor(random_state=99)

param_grid4 = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 1.0]
}

grid_search4 = GridSearchCV(estimator=rf_single4, param_grid=param_grid4, 
                           cv=5, n_jobs=-1, scoring='neg_root_mean_squared_error')

print("Starting hyperparameter tuning...")
grid_search4.fit(X4_train, y4_train)

best_rf_model4 = grid_search4.best_estimator_

print("\n--- Tuning Complete ---")
print(f"Best Hyperparameters: {grid_search4.best_params_}")
print("")
best_rf_model4.fit(X4_train, y4_train)

# Predict and Evaluate
predictions4 = best_rf_model4.predict(X4_test)
print("Predicted values for y_test")
print(predictions4)

# Calculate the error
mse4 = mean_squared_error(y4_test, predictions4)
rmse4 = np.sqrt(mse4)
print("")
print(f"RMSE for Curve Midpoint: {rmse4:.4f}")
print("-------------------------------------")

In [ ]:
print(predictions1)
print(predictions2)
print(predictions3)
print(predictions4)

MSE is decent for Growth Rate, but for others... it's still pretty innaccurate. Main bottleneck is I think just lack of data (trained on only 40 examples)

With Random State 99 = predicting on indices: 25, 36, 29, 22, 28, 31, 26, 13
Which are project codes: 5540, 5659, 5570, 5530, 5566, 5583, 5551, 5456



In [ ]:
# 1. Define the list of Project Codes in the exact order they match your predictions
project_codes = [5540, 5659, 5570, 5530, 5566, 5583, 5551, 5456]

# 2. Construct the DataFrame using a dictionary
# We map each column name (the key) to its corresponding array (the value)
predictions_df = pd.DataFrame({
    "Project Code": project_codes,
    "Duration_Months": predictions1,
    "S_Curve_L": predictions2,
    "S_Curve_k": predictions3,
    "S_Curve_t0": predictions4
})

# 3. Inspect the final DataFrame
print("Successfully created Predictions DataFrame:")
display(predictions_df) # Use display() in Jupyter notebooks for a nicely formatted table

In [ ]:
from matplotlib.ticker import MultipleLocator, FuncFormatter

# Define the currency formatter function before the loop
def format_currency(x, pos):
    """Formats large numbers into readable currency (e.g., $1.5M or $500K)"""
    if x >= 1e6:
        return f'${x * 1e-6:.1f}M'
    elif x >= 1e3:
        return f'${x * 1e-3:.0f}K'
    else:
        return f'${x:,.0f}'

# Loop through and plot projects
for index, row in predictions_df.head(60).iterrows():
    L = row['S_Curve_L']
    k = row['S_Curve_k']
    t0 = row['S_Curve_t0']
    duration = row['Duration_Months']
    
    # Generate X axis
    x_data = np.arange(int(duration))
    predicted_y = logistic_curve(x_data, L, k, t0)
    
    # Plot the curve
    plt.figure(figsize=(9, 4))
    plt.plot(x_data, predicted_y, linewidth=3, color='#2ca02c', label="Parameterized S-Curve")
    plt.title(f"Standardized S-Curve: Project {row['Project Code']}", fontsize=14)
    plt.xlabel("Months from Project Start", fontsize=12) 
    plt.ylabel("Cumulative Cost", fontsize=12) # Removed ($) since the formatter adds it
    
    ax = plt.gca() 
    
    # X-Axis: Major ticks every 3 months, minor ticks every 1 month
    ax.xaxis.set_major_locator(MultipleLocator(3))
    ax.xaxis.set_minor_locator(MultipleLocator(1))
    
    # Y-Axis: Format as currency, create ~8 even major tick slices
    ax.yaxis.set_major_formatter(FuncFormatter(format_currency))
    ax.yaxis.set_major_locator(plt.MaxNLocator(8)) 
    ax.yaxis.set_minor_locator(plt.MaxNLocator(16)) 
    
    # Update grid to show both major (solid) and minor (dashed) lines
    plt.grid(which='major', color='#CCCCCC', linestyle='-', linewidth=0.8)
    plt.grid(which='minor', color='#EEEEEE', linestyle='--', linewidth=0.5)
    
    plt.legend()
    plt.tight_layout()
    plt.show()

RESULT: Curves are fairly accurate! Duration is not too bad either!

Cumulative cost figures only are wayyy off. (culprit is S_Curve_L)

### NEXT THINGS TO TRY:
- Try a version (make a fresh copy of the book) with only cost data (no sq footage): -> PROS: MORE data, maybe more focused, CONS: less user input (loss of a key feature)

- Use the "Projected Budget" input to replace "Cumulative Cost" (makes sense... this is a better indicator than whatever the model could produce anyway)

- Use models to write a simple script where the user does the inputs (either with gross sq footage or without) and the S-Curve is created